In [ ]:
## Loading and setting up the data
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

DATA_DIR = "data2019"

# -----------------------------------------
# Load definitions once
# -----------------------------------------
sys.path.append(DATA_DIR)
import definitions_2019 as bd

# -----------------------------------------
# Load day once
# -----------------------------------------
DAY = 4
traj_file = os.path.join(DATA_DIR, f"beetrajectories_{DAY:03d}.hdf")
df = pd.read_hdf(traj_file)

current_date = bd.startday + pd.Timedelta(days=DAY)
print("Experiment starts on:", bd.startday)
print("Current date:", current_date)

# -----------------------------------------
# UID -> age via daydatamat.csv
# -----------------------------------------
cohort_df = pd.read_csv("daydatamat.csv")
cohort_df = cohort_df[cohort_df["Day number"] == DAY]

uid_to_age = dict(zip(cohort_df["Bee unique ID"], cohort_df["Age"]))

df["age(days)"] = df["uid"].map(uid_to_age)

# -----------------------------------------
# Sort for frame-sequential access
# -----------------------------------------
df = df.sort_values(["framenum", "uid"]).reset_index(drop=True)
df.loc[df["camera"] == 0, "x"] += bd.xpixels

frame_min = int(df["framenum"].min())
frame_max = int(df["framenum"].max())
print("Frame range:", frame_min, "to", frame_max)
print("Bees loaded:", df["uid"].nunique())
print("Age groups: ", sorted(df["age(days)"].dropna().unique()))

In [ ]:
## Finding out the framegaps for each bee, looking at the distribution of length of these fags and filling  in the MAX_GAP frames linearly

import numpy as np
import pandas as pd
import gc

MAX_GAP = 5

# ======================================================================
# Gap statistics
# ======================================================================

gap_lengths = []

for _, grp in df.groupby("uid", sort=False):
    frames = grp["framenum"].to_numpy()
    gaps = np.diff(frames) - 1
    gap_lengths.extend(gaps[gaps > 0])

gap_lengths = np.asarray(gap_lengths)

gap_counts = (
    pd.Series(gap_lengths)
      .value_counts()
      .sort_index()
      .rename_axis("Gap length")
      .reset_index(name="Count")
)

gap_counts["Fraction"] = gap_counts["Count"] / gap_counts["Count"].sum()
gap_counts["Cumulative"] = gap_counts["Fraction"].cumsum()

print(gap_counts)

print("\nSummary:")
for t in [1, 2, 3, 5]:
    frac = gap_counts.loc[gap_counts["Gap length"] <= t, "Fraction"].sum()
    print(f"Gap ≤ {t:2d}: {100*frac:6.2f}% of all gaps")

# ======================================================================
# Interpolate gaps up to MAX_GAP frames — vectorised
# ======================================================================

df = df.sort_values(["uid", "framenum"]).reset_index(drop=True)

# Compute per-row gap to next row within same uid
uid_arr    = df["uid"].to_numpy()
frame_arr  = df["framenum"].to_numpy()
x_arr      = df["x"].to_numpy(dtype=float)
y_arr      = df["y"].to_numpy(dtype=float)
theta_arr  = df["theta"].to_numpy(dtype=float)

# Mask: same uid, gap in [1, MAX_GAP]
same_uid   = uid_arr[:-1] == uid_arr[1:]
gap_arr    = frame_arr[1:] - frame_arr[:-1] - 1   # gap between row i and i+1
fill_mask  = same_uid & (gap_arr >= 1) & (gap_arr <= MAX_GAP)
fill_idx   = np.where(fill_mask)[0]   # indices i where we need to interpolate

print(f"\nGaps to fill: {len(fill_idx):,}")

if len(fill_idx) > 0:
    # For each gap, expand into (gap) interpolated rows
    # gap_sizes[k] = number of rows to insert for fill_idx[k]
    gap_sizes  = gap_arr[fill_idx]                   # shape (n_gaps,)
    total_rows = gap_sizes.sum()

    # Repeat each gap-start index gap_sizes[k] times
    rep_idx    = np.repeat(fill_idx, gap_sizes)      # which row i each interp row belongs to
    # j within each gap: 1, 2, ..., gap_size
    j_within   = np.ones(total_rows, dtype=int)
    pos = 0
    for k, gs in enumerate(gap_sizes):
        j_within[pos:pos+gs] = np.arange(1, gs+1)
        pos += gs

    # Faster: build j_within without Python loop using repeat + cumsum trick
    # (replace the loop above)
    counts     = gap_sizes
    j_within   = np.arange(total_rows) - np.repeat(
        np.concatenate(([0], np.cumsum(counts[:-1]))), counts
    ) + 1

    frac       = j_within / (gap_arr[rep_idx] + 1)  # interpolation fraction

    # Angular interpolation (shortest path)
    dtheta     = (theta_arr[rep_idx + 1] - theta_arr[rep_idx] + np.pi) % (2*np.pi) - np.pi

    interp_frame = frame_arr[rep_idx] + j_within
    interp_x     = x_arr[rep_idx]     + frac * (x_arr[rep_idx + 1]     - x_arr[rep_idx])
    interp_y     = y_arr[rep_idx]     + frac * (y_arr[rep_idx + 1]     - y_arr[rep_idx])
    interp_theta = theta_arr[rep_idx] + frac * dtheta
    interp_theta = (interp_theta + np.pi) % (2*np.pi) - np.pi

    # Build interpolated dataframe — inherit all other columns from row i
    interp_df = df.iloc[rep_idx].copy().reset_index(drop=True)
    interp_df["framenum"] = interp_frame
    interp_df["x"]        = interp_x
    interp_df["y"]        = interp_y
    interp_df["theta"]    = interp_theta

    print(f"Adding {len(interp_df):,} interpolated rows...")

    df = pd.concat([df, interp_df], ignore_index=True)
    df.sort_values(["uid", "framenum"], inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"Final dataframe size: {len(df):,} rows")

# ======================================================================
# Cleanup
# ======================================================================
del gap_lengths, gap_counts
gc.collect()

In [ ]:
##Loading the comb image
import displayfunctions as bp
import pickle
import gzip

zfilln = 3
comb_contents_dir = 'comb-contents-images2019/'
comb = pickle.load(gzip.open(comb_contents_dir+'comb_'+str(DAY).zfill(zfilln)+'.pklz','rb'))

In [ ]:
#Cleaning the temp files created; run before rerunning the animation cell
import shutil
shutil.rmtree("chunks", ignore_errors=True)
shutil.rmtree("slices", ignore_errors=True)

In [ ]:
##Quiver
##Main animation code
import os
import pickle
import subprocess
import numpy as np

START_FRAME = frame_min
END_FRAME   = frame_max
BOX_X          = (100, 6500)
BOX_Y          = (100, 1820)
FPS = 25

df2 = df.copy()
df2.loc[df2["camera"] == 0, "x"] += bd.xpixels

df_range = df2[
    (df2["framenum"].between(START_FRAME, END_FRAME))## &
    #(df2["x"].between(*BOX_X)) &
    #(df2["y"].between(*BOX_Y))
].copy()

total_frames   = END_FRAME - START_FRAME + 1 
#target_minutes = 180

# ── Config ───────────────────────────────────────────────────────────────────
N_WORKERS   = 14
STRIDE      = 1
FRAME_LIMIT = None
OUTPUT_DIR  = "chunks"
SLICES_DIR  = "slices"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SLICES_DIR, exist_ok=True)

# ── Split frames ─────────────────────────────────────────────────────────────
all_frames = list(range(START_FRAME, END_FRAME + 1, STRIDE))

if FRAME_LIMIT is not None:
    all_frames = all_frames[:FRAME_LIMIT]

frame_chunks = [c.tolist() for c in np.array_split(all_frames, N_WORKERS)]
print(f"Total frames: {len(all_frames)} | Workers: {N_WORKERS}")

# ── Age colors ───────────────────────────────────────────────────────────────
age_list = sorted(df["age(days)"].dropna().unique())

RED_SAFE_COLORS = [
    "#00FFFF",  # cyan
    "#FFFFFF",  # white
    "#39FF14",  # neon lime
    "#FF00FF",  # magenta
    "#FFD700",  # gold
    "#00BFFF",  # deep sky blue
    "#FF69B4",  # hot pink
]
age_colors = {a: RED_SAFE_COLORS[i % len(RED_SAFE_COLORS)] for i, a in enumerate(age_list)}

# ── Pickle each slice ────────────────────────────────────────────────────────
slice_paths = []
for i, frame_chunk in enumerate(frame_chunks):
    df_slice   = df_range[df_range["framenum"].isin(frame_chunk)]
    slice_path = os.path.join(SLICES_DIR, f"slice_{i:04d}.pkl")

    with open(slice_path, "wb") as f:
        pickle.dump({
            "df_slice"     : df_slice,
            "frame_indices": frame_chunk,
            "age_colors"   : age_colors,
            "age_list"     : age_list,
            "xpixels"      : bd.xpixels,
            "DAY"          : DAY,
            "comb"         : comb,
            "FPS"          : 25
        }, f)

    slice_paths.append(slice_path)
    print(f"  Slice {i}: {len(df_slice)} rows, {len(frame_chunk)} frames → {slice_path}")

print( "Minutes->", len(all_frames) / (FPS * 60))

# ── Launch all workers in parallel ───────────────────────────────────────────
procs = []
for i, slice_path in enumerate(slice_paths):
    cmd = [sys.executable, "worker_quiv.py", str(i), slice_path, OUTPUT_DIR]
    p   = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    procs.append((i, p))
    print(f"Launched worker {i} (pid {p.pid})")

# ── Stream output from all workers ───────────────────────────────────────────
import threading

def stream_output(chunk_id, proc):
    for line in proc.stdout:
        print(line, end="", flush=True)

threads = [threading.Thread(target=stream_output, args=(i, p)) for i, p in procs]
for t in threads: t.start()
for t in threads: t.join()

for i, p in procs:
    p.wait()
    print(f"Worker {i} exited with code {p.returncode}")

# ── Merge chunks ─────────────────────────────────────────────────────────────
chunk_paths = sorted([os.path.join(OUTPUT_DIR, f) for f in os.listdir(OUTPUT_DIR) if f.endswith(".mp4")])

list_file = "chunk_list.txt"
with open(list_file, "w") as f:
    for path in chunk_paths:
        f.write(f"file '{os.path.abspath(path)}'\n")

subprocess.run([
    "ffmpeg", "-y", "-f", "concat", "-safe", "0",
    "-i", list_file, "-c", "copy", f"bee_animation_{START_FRAME}_{END_FRAME}_quiv_day{DAY}_full.mp4"
], check=True)

os.remove(list_file)
print(f"bee_animation_{START_FRAME}_{END_FRAME}_quiv_day{DAY}_full.mp4")

In [ ]:
## Looking at the box we've set up. Easy to modify it from here.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Box coordinates
BOX_X = (100, 6500)
BOX_Y = (100, 2720)

# Draw comb
ax = bp.showcomb(comb)

# Draw box
rect = patches.Rectangle(
    (BOX_X[0], BOX_Y[0]),                 # bottom-left corner
    BOX_X[1] - BOX_X[0],                  # width
    BOX_Y[1] - BOX_Y[0],                  # height
    linewidth=3,
    edgecolor='red',
    facecolor='none'
)
ax.add_patch(rect)

ax.set_aspect('equal')
plt.show()